In [ ]:
!wget https://www.lamsade.dauphine.fr/~cazenave/project2025.zip
!unzip project2025.zip
!ls -l

!pip uninstall -y tensorflow
!pip install tensorrt-bindings==8.6.1
!pip install --extra-index-url https://pypi.nvidia.com tensorrt-libs
!pip install tensorflow[and-cuda]==2.15.0

# redemarrer la session

--2025-04-02 14:18:28--  https://www.lamsade.dauphine.fr/~cazenave/project2025.zip
Resolving www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)... 193.48.71.250
Connecting to www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)|193.48.71.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 486727746 (464M) [application/zip]
Saving to: ‘project2025.zip’

project2025.zip     100%[===================>] 464.18M  74.6MB/s    in 6.3s    

2025-04-02 14:18:35 (73.9 MB/s) - ‘project2025.zip’ saved [486727746/486727746]

Archive:  project2025.zip
  inflating: games2022.data          
  inflating: Golois.cpython-310-x86_64-linux-gnu.so  
  inflating: Golois.cpython-38-x86_64-linux-gnu.so  
  inflating: golois.cpython-310-x86_64-linux-gnu.so  
  inflating: golois.cpython-311-x86_64-linux-gnu.so  
  inflating: golois.cpython-37m-x86_64-linux-gnu.so  
  inflating: golois.cpython-38-x86_64-linux-gnu.so  
  inflating: trainGolois.py          
total 2382016
-rw-r--r-- 1 root root 

In [ ]:
# Import Libraries

import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import gc
import golois

print ("Tensorflow version", tf.__version__)

Tensorflow version 2.15.0


In [ ]:
# Configuration

planes = 31
moves = 361
N = 10000


input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')

In [ ]:
# Get Validation Data

print ("getValidation", flush = True)
golois.getValidation (input_data, policy, value, end)

getValidation


In [ ]:
from keras.layers import Input, Dense, Conv2D, Flatten, BatchNormalization, Activation, LeakyReLU, add
from keras.optimizers import SGD
from keras import regularizers

# Model Creation
#
# Architecture simple de type AlphaGo-style, avec deux têtes :
# une policy (pour prédire un coup) et une value (pour prédire l’issue de la partie).
#
# Policy - Prédit le prochain coup. Valeur entre 1 et 361
#          Utilise la Categorical Cross Entropy.
# Value  - Prédit la probabilité de victoire. Valeur entre 0 et 1

nb_filters = 64
filter_size = 3
block_iteration=5
dropout_rate = 0.5

l2_reg = 0.0001

def get_iteration_block(input, nb_filters=nb_filters, filter_size=filter_size):
  x = layers.Conv2D(nb_filters, filter_size, padding='same')(input)
  x = BatchNormalization(axis=1)(x)
  x = layers.Conv2D(nb_filters, filter_size, padding='same')(x)
  x = BatchNormalization(axis=1)(x)
  x = add([input, x])
  x = layers.LeakyReLU()(x)
  return x

def get_model(nb_filters=nb_filters, filter_size=filter_size, l2_reg = l2_reg):

  input = keras.Input(shape=(19, 19, planes), name='board')
  x = layers.Conv2D(nb_filters, 1, padding='same')(input)
  x = BatchNormalization(axis=1)(x)
  x = layers.LeakyReLU()(x)

  for i in range (block_iteration):
     x = get_iteration_block(x, nb_filters, filter_size)

  policy_head = layers.Conv2D(1, 1, padding='same', use_bias = False, kernel_regularizer=regularizers.l2(l2_reg))(x)
  policy_head = BatchNormalization(axis=1)(policy_head)
  policy_head = layers.LeakyReLU()(policy_head)
  policy_head = layers.Flatten()(policy_head)
  policy_head = layers.Activation('softmax', name='policy')(policy_head)

  value_head = layers.Conv2D(1, 1, padding='same', use_bias = False, kernel_regularizer=regularizers.l2(l2_reg))(x)
  value_head = BatchNormalization(axis=1)(value_head)
  value_head = layers.LeakyReLU()(value_head)
  value_head = layers.MaxPool2D(pool_size=(2, 2))(value_head)
  value_head = layers.Flatten()(value_head)
  value_head = layers.Dense(50, kernel_regularizer=regularizers.l2(l2_reg))(value_head)
  value_head = BatchNormalization(axis=1)(value_head)
  value_head = layers.LeakyReLU()(value_head)
  value_head = layers.Dense(1, activation='sigmoid', name='value', kernel_regularizer=regularizers.l2(l2_reg))(value_head)

  model = keras.Model(inputs=input, outputs=[policy_head, value_head])
  return model

model = get_model(nb_filters, filter_size, l2_reg)
model.summary ()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 board (InputLayer)          [(None, 19, 19, 31)]         0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 19, 19, 32)           1024      ['board[0][0]']               
                                                                                                  
 batch_normalization (Batch  (None, 19, 19, 32)           76        ['conv2d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 leaky_re_lu (LeakyReLU)     (None, 19, 19, 32)           0         ['batch_normalization[0][0

In [ ]:
from tensorflow.keras.callbacks import Callback
import keras.backend as K
import math

class EpochCyclicLRScheduler(Callback):
    def __init__(self, base_lr=1e-4, max_lr=0.5, step_size=10, mode='triangular'):
        super().__init__()
        self.base_lr = base_lr
        self.max_lr = max_lr
        self.step_size = step_size
        self.iterations = 0
        self.history = {}
        self.mode = mode

    def clr(self):
        cycle = math.floor(1 + self.iterations / (2 * self.step_size))
        x = abs(self.iterations / self.step_size - 2 * cycle + 1)
        scale_fn = {
            'triangular': lambda: 1.0,
            'triangular2': lambda: 1 / (2. ** (cycle - 1)),
            'exp_range': lambda: 0.95 ** self.iterations
        }.get(self.mode, lambda: 1.0)
        return self.base_lr + (self.max_lr - self.base_lr) * max(0, (1 - x)) * scale_fn()

    def on_epoch_begin(self, epoch, logs=None):
        lr = self.clr()
        K.set_value(self.model.optimizer.lr, lr)
        self.history.setdefault('lr', []).append(lr)
        print(f"[CLR] Epoch {epoch + 1} — LR set to: {lr:.6f}")
        self.iterations += 1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import time
from keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, Callback
from datetime import datetime

# Configuration
epochs = 500
batch_size = 256

# loss_weights : pondère la contribution des deux sorties à la loss totale
# - policy_weight : importance de la politique (jouer la bonne case)
# - value_weight : importance de la valeur (prédire le gagnant)
policy_weight = 1.0
value_weight = 1.0

# Stochastic Gradient Descent (SGD) avec un taux d’apprentissage et un momentum personnalisés
learning_rate=0.5
momentum=0.9

optimizer_sgd = keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum, nesterov=False)
optimizer_nesterov = keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum, nesterov=True)
optimizer_adagrad = keras.optimizers.Adagrad(learning_rate=learning_rate)
optimizer_adadelta = keras.optimizers.Adadelta(learning_rate=learning_rate)

def get_learning_rate_scheduler_callback():
    return ReduceLROnPlateau(
        monitor='loss',          # Peut être 'val_loss' si tu veux évaluer sur validation
        factor=0.5,                                     # Réduction de LR (50%)
        patience=3,                                     # Attendre 5 epochs sans amélioration
        min_delta=0.005,
        min_lr=1e-6,                                    # Ne descend jamais en dessous
        cooldown=2,                                     # Rétablir LR après 5 epochs
        verbose=1
    )

def get_model_checkpoint_callback():
  # Créer un horodatage
  best_model_name = f"best_mchettih.h5"

  # Callback ModelCheckpoint avec horodatage dans le nom du fichier
  checkpoint = ModelCheckpoint(
      filepath=best_model_name,
      monitor='loss',           # ou 'val_policy_categorical_accuracy', etc.
      save_best_only=True,
      save_weights_only=False,
      verbose=1
  )
  return checkpoint

def get_cyclic_lr_scheduler_callback():
  cyclic_lr_callback = EpochCyclicLRScheduler(
      base_lr=1e-4,
      max_lr=0.05,        # Réduit pour ne pas dépasser un bon range
      step_size=20,       # Un cycle complet = 4 epochs (2 aller + 2 retour)
      mode='triangular2'  # Diminue progressivement l’amplitude
  )
  return cyclic_lr_callback

def compile_model(model, optimizer, policy_weight=policy_weight, value_weight=value_weight, batch_size=batch_size, value=value):

  model.compile(optimizer=optimizer,
    loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
    loss_weights={'policy' : policy_weight, 'value' : value_weight},
    metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

  # Pour stocker les métriques de chaque epoch
  all_history = []

  # Démarrer le chrono
  start_time = time.time()

  # Récupère le learning rate scheduler callback
  learning_rate_scheduler_callback = get_learning_rate_scheduler_callback()
  # Récupére le best model checkpoint callback
  model_checkpoint_callback = get_model_checkpoint_callback()
  #Récupere cyclic_lr_callback
  cyclic_lr_callback = get_cyclic_lr_scheduler_callback()

  for i in range(1, epochs + 1):
      optimizer_name = model.optimizer.__class__.__name__
      learning_rate = float(keras.backend.get_value(model.optimizer.lr))
      print(f"{optimizer_name} - Epoch {i}, Scheduler LR {learning_rate}")

      golois.getBatch(input_data, policy, value, end, groups, i * N)

      cyclic_lr_callback.set_model(model)
      cyclic_lr_callback.on_epoch_begin(i - 1)

      history = model.fit(input_data, # Input
          {'policy': policy, 'value': value}, #Output
          epochs=1,
          batch_size=batch_size,
          verbose=1)

      # Stocker l’historique
      metrics = {key: val[0] for key, val in history.history.items()}

      model_checkpoint_callback.set_model(model)
      #import pdb; pdb.set_trace()
      model_checkpoint_callback.on_epoch_end(i - 1, metrics)

      metrics['epoch'] = i
      all_history.append(metrics)


      if i % 5 == 0:
          gc.collect()

      if i % epochs == 0:
          golois.getValidation(input_data, policy, value, end)
          val = model.evaluate(input_data, [policy, value], verbose=0, batch_size=batch_size)

          total_time = time.time() - start_time
          minutes, seconds = divmod(total_time, 60)
          return val, pd.DataFrame(all_history), total_time

def plot_result(history_dfs, labels, epochs=None, learning_rate=None, batch_size=None):
    assert len(history_dfs) == len(labels)

    # Titre
    info = []
    if epochs: info.append(f"Epochs: {epochs}")
    if learning_rate: info.append(f"Learning Rate: {learning_rate}")
    if batch_size: info.append(f"Batch Size: {batch_size}")
    title = "Training Summary" + (" — " + ", ".join(info) if info else "")

    # Grille personnalisée : 2 lignes (3 en haut, 2 en bas)
    fig = plt.figure(figsize=(18, 8))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 3)

    # --- Ligne 1 : 3 plots ---
    ax1 = fig.add_subplot(gs[0, 0])
    for df, label in zip(history_dfs, labels):
        ax1.plot(df['epoch'], df['loss'], label=f'{label} Total Loss')
    ax1.set_title('Total Loss par Epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Total Loss')
    ax1.legend()

    ax2 = fig.add_subplot(gs[0, 1])
    for df, label in zip(history_dfs, labels):
        if 'policy_loss' in df.columns:
            ax2.plot(df['epoch'], df['policy_loss'], label=f'{label} Policy Loss')
    ax2.set_title('Policy Loss par Epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Policy Loss')
    ax2.legend()

    ax3 = fig.add_subplot(gs[0, 2])
    for df, label in zip(history_dfs, labels):
        if 'value_loss' in df.columns:
            ax3.plot(df['epoch'], df['value_loss'], label=f'{label} Value Loss')
    ax3.set_title('Value Loss par Epoch')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Value Loss')
    ax3.legend()

    # --- Ligne 2 : 2 plots ---
    ax4 = fig.add_subplot(gs[1, 0])
    for df, label in zip(history_dfs, labels):
        if 'policy_categorical_accuracy' in df.columns:
            ax4.plot(df['epoch'], df['policy_categorical_accuracy'], label=label)
    ax4.set_title('Policy Accuracy par Epoch')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Categorical Accuracy')
    ax4.legend()

    ax5 = fig.add_subplot(gs[1, 1])
    for df, label in zip(history_dfs, labels):
        if 'value_mse' in df.columns:
            ax5.plot(df['epoch'], df['value_mse'], label=label)
    ax5.set_title('Value MSE par Epoch')
    ax5.set_xlabel('Epoch')
    ax5.set_ylabel('MSE')
    ax5.legend()

    # Libérer la dernière case vide de la 2e ligne
    fig.delaxes(fig.add_subplot(gs[1, 2]))  # case vide propre

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

def print_validation_results(model_results, epoch=epochs):
    """
    Affiche les résultats de validation pour une liste de modèles.

    Paramètres :
    - model_results : liste de tuples (model, val, label)
    - epoch : (optionnel) numéro d'epoch à afficher dans le titre
    """
    for model, val, label, time in model_results:
        metrics = dict(zip(model.metrics_names, val))
        title = f"📊 Validation Results for {label}"
        if epoch is not None:
            title += f" — Epoch {epoch}"
        print(f"\n{title}:")
        for name, value in metrics.items():
            print(f"  - {name:<30}: {value:.4f}")
        print(f"  - Time: {time:.4f}")

# Model Compilation and Train
#model_sgd = get_model(nb_filters, filter_size, l2_reg)
#val_sgd, all_sgd_history, total_sgd_time = compile_model(model_sgd, optimizer_sgd)

model_nesterov = get_model(nb_filters, filter_size, l2_reg)
val_nesterov, all_nesterov_history, total_nesterov_time = compile_model(model_nesterov, optimizer_nesterov)

#model_adagrad = get_model(nb_filters, filter_size, l2_reg)
#val_adagrad, all_adagrad_history, total_adagrad_time = compile_model(model_adagrad, optimizer_adagrad)

#model_adadelta = get_model(nb_filters, filter_size, l2_reg)
#val_adadelta, all_adadelta_history, total_adadelta_time = compile_model(model_adadelta, optimizer_adadelta)

# Affichage des résultats
results = [
    #(model_sgd, val_sgd, "SGD", total_sgd_time),
    (model_nesterov, val_nesterov, "Nesterov", total_nesterov_time)#,
    #(model_adagrad, val_adagrad, "Adagrad", total_adagrad_time),
    #(model_adadelta, val_adadelta, "Adadelta", total_adadelta_time)
]
print_validation_results(results)

# Affichage des courbes comparatives
plot_result(
    #history_dfs=[all_sgd_history, all_nesterov_history, all_adagrad_history, all_adadelta_history],
    history_dfs=[all_nesterov_history],
    #labels=["SGD", "Nesterov", "Adagrad", "Adadelta"],
    abels=["Nesterov"],
    epochs=epochs,
    learning_rate=learning_rate,
    batch_size=batch_size
)

SGD - Epoch 1, Scheduler LR 0.5
[CLR] Epoch 1 — LR set to: 0.000100
40/40 [==============================] - 12s 61ms/step - loss: 6.8688 - policy_loss: 6.0647 - value_loss: 0.7972 - policy_categorical_accuracy: 0.0043 - value_mse: 0.1578

Epoch 1: loss improved from inf to 6.86878, saving model to best_mchettih.h5
SGD - Epoch 2, Scheduler LR 9.999999747378752e-05


/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


[CLR] Epoch 2 — LR set to: 0.002595
40/40 [==============================] - 2s 58ms/step - loss: 6.1771 - policy_loss: 5.4266 - value_loss: 0.7437 - policy_categorical_accuracy: 0.0199 - value_mse: 0.1420

Epoch 2: loss improved from 6.86878 to 6.17713, saving model to best_mchettih.h5
SGD - Epoch 3, Scheduler LR 0.0025949999690055847
[CLR] Epoch 3 — LR set to: 0.005090
40/40 [==============================] - 2s 53ms/step - loss: 5.4064 - policy_loss: 4.6962 - value_loss: 0.7033 - policy_categorical_accuracy: 0.1072 - value_mse: 0.1256

Epoch 3: loss improved from 6.17713 to 5.40636, saving model to best_mchettih.h5
SGD - Epoch 4, Scheduler LR 0.005090000107884407
[CLR] Epoch 4 — LR set to: 0.007585
40/40 [==============================] - 2s 55ms/step - loss: 4.9090 - policy_loss: 4.2065 - value_loss: 0.6957 - policy_categorical_accuracy: 0.1658 - value_mse: 0.1227

Epoch 4: loss improved from 5.40636 to 4.90901, saving model to best_mchettih.h5
SGD - Epoch 5, Scheduler LR 0.0075849